In [0]:
from pyspark.sql import SparkSession
import os
import z_pipelineUtils as z_pU
import z_schemas as z_s
import z_tests as z_t

# File that stores the most recent run details of the pipeline
PIPELINE_STATE = "C:/Users/jedg1/OneDrive/Documents/Developer/state_voter_files/nc_dbricks/pipeline_state.json"
# Directory that will store the unzipped files
UNZIP_DEST = "C:/Users/jedg1/OneDrive/Documents/Developer/state_voter_files/nc_dbricks/unzipped"
# Encoding on the voter files
# FILE_ENCODING = "latin-1"
FILE_ENCODING = "iso-8859-1"

saved_state, live_state = z_pU.readStateJson(PIPELINE_STATE)
pre_tests = z_t.PRE_TESTS

spark = SparkSession.builder.getOrCreate()

for voter_file in saved_state:
    downloaded = saved_state[voter_file]["downloaded"]
    unzipped = saved_state[voter_file]["unzipped"]
    tested = saved_state[voter_file]["tested"]
    if (downloaded is True) and (unzipped is True) and (tested is False):
        data_filename = voter_file.replace(".zip", ".txt")
        full_file_path = os.path.join(UNZIP_DEST, data_filename)
        print(f"Loading {voter_file}...")
        v_df = spark.read.csv(full_file_path, sep="\t", encoding=FILE_ENCODING, quote='"', header=True)
        table_name = z_s.SCHEMAS[voter_file]["table_name"]
        print(f"Testing {voter_file}...")
        test_results = {}
        for test in pre_tests[table_name]:
            func = getattr(z_pU, test["function"])
            result = func(v_df, voter_file, **test["params"])
            test_results[test["function"]] = result
        if False not in test_results.values():
            print(f"All tests passed.")
            live_state[voter_file]["tested"] = True
            z_pU.updateStateJson(live_state, PIPELINE_STATE)
        else:
            print("Test(s) failed.")
            for func, result in test_results.items():
                print(f"{func}: {result}")
    elif tested is True:
        print(f"Newest dataset for {voter_file} has already been tested.")
    else:
        print(f"{voter_file} is not ready for testing (downloaded={downloaded}, unzipped={unzipped}).")
    print()
